# Setup

In [2]:
import sys
sys.path.append('../src/')  # Replace with your actual path
from train import str2bool, set_seed, parse_config, get_predictions, get_attention, get_embedding

import json

import datetime
import os
import pickle as pkl
import random

import numpy as np
import pandas as pd
import pytorch_lightning as pl
import torch
from pytorch_lightning.callbacks import ModelCheckpoint
from pytorch_lightning.callbacks.early_stopping import EarlyStopping

#from datamodule import  ReplogleDataModule
from presage_datamodule import ReploglePRESAGEDataModule
from model_harness import ModelHarness
from presage import PRESAGE


In [ ]:
dataset="replogle_rpe1_essential_unfiltered"
seed="seed_0"

default_config_file = "../configs/defaults_config.json"
singles_config_file = "../configs/singles_config.json"
ds_config_file = f"../configs/{dataset}_config.json"

# Load the default config
with open(default_config_file, "r") as f:
    config = json.load(f)
with open(singles_config_file,"r") as f:
    singles_config = json.load(f)
with open(ds_config_file,"r") as f:
    ds_config = json.load(f)

singles_config.update(singles_config)
singles_config.update(ds_config)

new_config = {}
for key, value in singles_config.items():
    if value is not None and key not in {"config", "data_config"}:
        new_config[key.replace("_", ".", 1)] = value
singles_config = new_config
config.update(singles_config)

In [4]:
modify_config = {"training.eval_test":False,
"model.pathway_files": "../sample_files/prior_files/sample.knowledge_experimental.txt",
"data.data_dir":"../data/",}

config.update(modify_config)

In [5]:
config = parse_config(config)

set_seed(config["training"].pop("seed", None))

offline = config["training"].pop("offline", False)
do_test_eval = config["training"].pop("eval_test", True)

In [6]:
predictions_file = config["training"].pop("predictions_file", None)
embedding_pref = config["training"].pop("embedding_file", None)
attention_file = config["training"].pop("attention_file", None)


# Initialize the data module

In [7]:
config['data']['dataset'] = dataset


config['data']['seed'] = f"../splits/{dataset}_random_splits/{seed}.json"


In [8]:
seed = config["data"].pop("seed")
datamodule = ReploglePRESAGEDataModule.from_config(config["data"])
datamodule.do_test_eval = do_test_eval

if hasattr(datamodule, "set_seed"):
    datamodule.set_seed(seed)
config["data"]["seed"] = seed

../data/


# Prepare Datamodule

In [9]:
datamodule.prepare_data()

datamodule.setup("fit")

print("datamodule setup complete.")

../data/replogle_k562_essential_unfiltered/perturb_processed.h5ad
Found local data file ../data/replogle_k562_essential_unfiltered/perturb_processed.h5ad
Found local extracted data file ../data/replogle_k562_essential_unfiltered/perturb_processed.h5ad
Found local preprocessed data file ../data/replogle_k562_essential_unfiltered/replogle_k562_essential_unfiltered_processed.h5ad
Found local preprocessed data file ../data/replogle_k562_essential_unfiltered/degs/merged.degs.json
Loading adata...
Loading DEGs...


/raid/yangpeng_lab/c12212609/PRESAGE/notebooks/../src/datamodule.py:37: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  ).pipe(lambda df: df.groupby(df.index).mean())


Computing pseudobulk...


/raid/yangpeng_lab/c12212609/PRESAGE/notebooks/../src/datamodule.py:37: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  ).pipe(lambda df: df.groupby(df.index).mean())


Computing pseudobulk...


/raid/yangpeng_lab/c12212609/PRESAGE/notebooks/../src/datamodule.py:37: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  ).pipe(lambda df: df.groupby(df.index).mean())
/raid/yangpeng_lab/c12212609/PRESAGE/notebooks/../src/datamodule.py:556: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  for tup in ad.obs.groupby(self.perturb_field)
/raid/yangpeng_lab/c12212609/PRESAGE/notebooks/../src/datamodule.py:556: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and

datamodule setup complete.


# Initialize PRESAGE

In [10]:
# initialize model
model_config = config["model"]
model_config["dataset"] = dataset

# legacy unused parameters
model_config['pca_dim'] = None
model_config['source'] = 'temp'
model_config['learnable_gene_embedding'] = False

module = PRESAGE(
    model_config,
    datamodule,
    datamodule.pert_covariates.shape[1],
    datamodule.n_genes,
    # latent_dim or datamodule.n_genes,
)

if hasattr(module, "custom_init"):
    module.custom_init()


Computing embeddings on Transpose matrix...


/raid/yangpeng_lab/c12212609/PRESAGE/notebooks/../src/presage.py:741: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  for tup in adata.obs.groupby(datamodule.perturb_field)


Computing embeddings on Coexpression...


# Set up model training

In [11]:
lightning_module = ModelHarness(
    module,
    datamodule,
    model_config,
)

print("model initialization complete.")

# run trainer
logger = pl.loggers.CSVLogger(
    save_dir="./logs",
    name=dataset,
    version=seed.split('/')[-1].split('.json')[0]
)

if predictions_file == "None":
    predictions_file = None

early_stop_callback = EarlyStopping(
    monitor="val_loss",
    min_delta=1e-6,
    patience=10,
    verbose=True,
    mode="min",
)

# Get current date and time
now = datetime.datetime.now()

# Format the date and time
now_str = now.strftime("%Y-%m-%d-%H-%M-%S")

checkpoint_callback = ModelCheckpoint(
    monitor="val_loss",
    dirpath="./saved_models",
    filename=f"my_model-{dataset}-{seed.split('/')[-1].split('.json')[0]}-{now_str}-{{epoch:02d}}-{{val_loss:.2f}}",
    save_top_k=1,
    mode="min",
)
torch.autograd.set_detect_anomaly(True)
trainer = pl.Trainer(
    logger=logger,
    log_every_n_steps=3,
    num_sanity_val_steps=10,
    callbacks=[early_stop_callback, checkpoint_callback],
    reload_dataloaders_every_n_epochs=1,
    **config["training"],
    gradient_clip_val=0.1,
)

Loading adata...
Loading DEGs...


/raid/yangpeng_lab/c12212609/PRESAGE/notebooks/../src/datamodule.py:37: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  ).pipe(lambda df: df.groupby(df.index).mean())
/raid/yangpeng_lab/c12212609/PRESAGE/notebooks/../src/evaluator.py:1161: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  grouped_adata = self.single_cells.obs.groupby("gene")
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs


model initialization complete.


# Fit PRESAGE and retain the best model

In [12]:
trainer.fit(lightning_module, datamodule=datamodule)
# lightning_module is the pytorch lighting, datamodule from datamodule.py
# Get the best model path
best_model_path = checkpoint_callback.best_model_path

LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15]

  | Name   | Type    | Params | Mode 
-------------------------------------------
0 | module | PRESAGE | 9.3 M  | train
-------------------------------------------
9.3 M     Trainable params
0         Non-trainable params
9.3 M     Total params
37.146    Total estimated model params size (MB)
224       Modules in train mode
0         Modules in eval mode


../data/replogle_k562_essential_unfiltered/perturb_processed.h5ad
Found local data file ../data/replogle_k562_essential_unfiltered/perturb_processed.h5ad
Found local extracted data file ../data/replogle_k562_essential_unfiltered/perturb_processed.h5ad
Found local preprocessed data file ../data/replogle_k562_essential_unfiltered/replogle_k562_essential_unfiltered_processed.h5ad
Found local preprocessed data file ../data/replogle_k562_essential_unfiltered/degs/merged.degs.json
Sanity Checking: |          | 0/? [00:00<?, ?it/s]

/raid/yangpeng_lab/c12212609/miniconda3/envs/topic/lib/python3.10/site-packages/pytorch_lightning/trainer/connectors/data_connector.py:433: The 'val_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=95` in the `DataLoader` to improve performance.


/raid/yangpeng_lab/c12212609/miniconda3/envs/topic/lib/python3.10/site-packages/pytorch_lightning/trainer/connectors/data_connector.py:433: The 'train_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=95` in the `DataLoader` to improve performance.


Epoch 0: 100%|██████████| 68/68 [00:09<00:00,  6.86it/s, v_num=ed_0]

...

Epoch 5: 100%|██████████| 68/68 [00:10<00:00,  6.58it/s, v_num=ed_0]

Metric val_loss improved by 0.060 >= min_delta = 1e-06. New best score: 1.361


Epoch 15: 100%|██████████| 68/68 [00:10<00:00,  6.64it/s, v_num=ed_0]

Monitored metric val_loss did not improve in the last 10 records. Best score: 1.361. Signaling Trainer to stop.


Epoch 15: 100%|██████████| 68/68 [00:10<00:00,  6.64it/s, v_num=ed_0]


# set up data module and run test set through trained PRESAGE

In [13]:
datamodule.setup("test")
datamodule._data_setup = False

checkpoint = torch.load(best_model_path)
lightning_module.load_state_dict(checkpoint["state_dict"])
os.remove(best_model_path)

# log final eval metrics
trainer.test(lightning_module, datamodule=datamodule)


dataloader = datamodule.test_dataloader()
avg_predictions = get_predictions(
    trainer, lightning_module, dataloader, datamodule.var_names
)
avg_predictions = avg_predictions.loc[
    :, datamodule.train_dataset.adata.var.measured_gene
]
avg_predictions.to_csv(predictions_file)

../data/replogle_k562_essential_unfiltered/perturb_processed.h5ad
Found local data file ../data/replogle_k562_essential_unfiltered/perturb_processed.h5ad
Found local extracted data file ../data/replogle_k562_essential_unfiltered/perturb_processed.h5ad
Found local preprocessed data file ../data/replogle_k562_essential_unfiltered/replogle_k562_essential_unfiltered_processed.h5ad
Found local preprocessed data file ../data/replogle_k562_essential_unfiltered/degs/merged.degs.json
Loading adata...
Loading DEGs...


/raid/yangpeng_lab/c12212609/PRESAGE/notebooks/../src/datamodule.py:37: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  ).pipe(lambda df: df.groupby(df.index).mean())


5
Computing pseudobulk...


/raid/yangpeng_lab/c12212609/PRESAGE/notebooks/../src/datamodule.py:37: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  ).pipe(lambda df: df.groupby(df.index).mean())
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15]
/raid/yangpeng_lab/c12212609/miniconda3/envs/topic/lib/python3.10/site-packages/pytorch_lightning/trainer/connectors/data_connector.py:433: The 'test_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=95` in the `DataLoader` to improve performance.


Testing DataLoader 0: 100%|██████████| 94/94 [00:03<00:00, 28.34it/s]20
200
Testing DataLoader 0: 100%|██████████| 94/94 [00:42<00:00,  2.23it/s]


┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃              Test metric               ┃              DataLoader 0              ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│       test_avg_cossim_top200_de        │            0.18903748691082            │
│     test_avg_cossim_top200_unionde     │          0.27913814783096313           │
│        test_avg_cossim_top20_de        │           0.1268380582332611           │
│     test_avg_cossim_top20_unionde      │           0.2827605605125427           │
│   test_avg_normalized_mse_top200_de    │           0.7488459944725037           │
│ test_avg_normalized_mse_top200_unionde │           0.8769591450691223           │
│    test_avg_normalized_mse_top20_de    │           0.743890643119812            │
│ test_avg_normalized_mse_top20_unionde  │           0.8734482526779175           │
│               test_loss                │           0.9586241245269775           │
└────────────────────────────────────────┴────────────────────────────────────────┘

LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15]


../data/replogle_k562_essential_unfiltered/perturb_processed.h5ad
Found local data file ../data/replogle_k562_essential_unfiltered/perturb_processed.h5ad
Found local extracted data file ../data/replogle_k562_essential_unfiltered/perturb_processed.h5ad
Found local preprocessed data file ../data/replogle_k562_essential_unfiltered/replogle_k562_essential_unfiltered_processed.h5ad
Found local preprocessed data file ../data/replogle_k562_essential_unfiltered/degs/merged.degs.json


/raid/yangpeng_lab/c12212609/miniconda3/envs/topic/lib/python3.10/site-packages/pytorch_lightning/trainer/connectors/data_connector.py:433: The 'predict_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=95` in the `DataLoader` to improve performance.


Predicting DataLoader 0: 100%|██████████| 94/94 [00:04<00:00, 22.71it/s]


In [14]:
avg_predictions

,LINC01409,LINC01128,NOC2L,KLHL17,HES4,ISG15,SDF4,B3GALT6,UBE2J2,ACAP3,...,MT-ND4L,MT-ND4,MT-ND5,MT-ND6,MT-CYB,BX004987.1,MAFIP,AL354822.1,AC240274.1,AC004556.3
perturbation,,,,,,,,,,,,,,,,,,,,,
AAAS,0.070946,0.034170,-0.107758,-0.093054,-0.034525,0.027895,-0.052143,-0.020438,-0.103033,-0.016308,...,0.088190,4.298061,0.291028,-0.113765,5.972149,-0.021196,0.011117,-0.072951,0.033752,0.006982
AAMP,0.023562,0.001603,-0.036282,-0.005864,-0.013554,0.010353,-0.081509,-0.005933,-0.000893,-0.035139,...,-0.252553,-3.795122,-0.658977,-0.479971,-0.607476,-0.069950,0.019609,-0.034877,0.068649,0.017702
AARS2,0.225857,-0.033994,-0.147723,-0.097233,-0.062811,0.051984,-0.179050,-0.056587,-0.085889,-0.085017,...,0.116487,12.607813,1.264133,0.193868,14.026630,0.023174,0.049788,-0.133210,0.180808,-0.006852
AATF,0.182455,-0.078042,-0.470555,-0.075650,0.169801,-0.011315,-0.260000,0.158267,-0.019853,-0.021623,...,-2.725441,-31.803934,-5.810283,-2.729616,-12.892108,-0.063320,0.021399,0.182131,0.198694,0.063479
ABCB10,-0.006056,-0.015184,-0.120992,-0.031819,0.072068,0.012412,-0.082091,0.016612,-0.080898,-0.044464,...,-0.678874,-6.491302,-1.285042,-0.777435,-1.549224,-0.029179,-0.009827,-0.008865,0.022738,0.027657
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
ZNF763,0.196319,-0.022610,-0.107740,-0.110387,-0.071738,0.054546,-0.163293,-0.061118,-0.080859,-0.105977,...,0.235687,12.542147,1.249450,0.206917,13.781375,0.012992,0.036147,-0.121754,0.179877,-0.023948
ZNHIT2,0.014542,0.012596,-0.110146,-0.028378,0.098919,-0.045224,-0.059598,0.033865,-0.055959,-0.029304,...,-0.625779,-7.087514,-1.297239,-0.728606,-2.245303,-0.021761,-0.001296,0.001745,0.023485,0.012852
ZNHIT3,0.056894,-0.012299,0.007495,-0.040526,0.023478,0.051388,-0.042518,-0.018811,-0.013498,-0.027402,...,-0.225007,-1.581409,-0.373823,-0.406722,0.809181,0.008662,0.007646,-0.020339,0.016220,0.002010
